In [4]:
import torch
import torch.nn as nn
import torchvision 
import torch.optim as optim
from torchvision.datasets import CIFAR10
from torch.utils.data import TensorDataset, DataLoader
import torchvision.transforms as transforms

In [5]:
#Datasets and dataloaders
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", download=True, train=True, transform=transform)
testset = CIFAR10(root="./data", download=False, train=True, transform=transform)

100%|██████████| 170M/170M [05:53<00:00, 482kB/s]  


In [7]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

Build CNN

In [9]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, padding=1, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel =2 stride=2

            nn.Conv2d(32, 64, padding=1, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2), 

            nn.Conv2d(64, 128, padding=1, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) #flattening
        x = self.fc_layers(x)
        return x


In [10]:
model = CNN()
optimiser = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

Train the Model

In [13]:
epochs = 10
best_val_loss = float("inf")

for epoch in range(epochs):
    epoch_training_loss = 0.0
    epoch_val_loss = 0.0
    model.train()
    
    for images, labels in trainloader:
        optimiser.zero_grad()
        outputs = model.forward(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()
        epoch_training_loss+=loss

    print(f"epoch = {epoch} ==> train loss = {epoch_training_loss/len(trainloader)}")

epoch = 0 ==> train loss = 0.4334159791469574
epoch = 1 ==> train loss = 0.3380209803581238
epoch = 2 ==> train loss = 0.2648775577545166
epoch = 3 ==> train loss = 0.20286864042282104
epoch = 4 ==> train loss = 0.15989543497562408
epoch = 5 ==> train loss = 0.13237936794757843
epoch = 6 ==> train loss = 0.1040271446108818
epoch = 7 ==> train loss = 0.0931025967001915
epoch = 8 ==> train loss = 0.09829016029834747
epoch = 9 ==> train loss = 0.08410026878118515


In [ ]:
correct = 0
total = 0

with torch.no_grad():
    model.eval()
    for images, labels in testloader:
        output = model.forward(images)
        _, predicted = torch.max(output, 1)
        correct += (predicted==labels).sum().item()
        total += labels.size(0)


accuracy = 98.234


In [16]:
print(f"accuracy = {correct/total*100} %")

accuracy = 98.234 %
